# USP at AmericasNLP 2026 — Stage 2: NMT Fine-tuning

Fine-tunes NLLB-200-distilled-600M (Spanish → Indigenous language) for each language.

**Two platforms used:**
- Wixárika, Nahuatl, Bribri, Guaraní: Google Colab Pro (A100)
- **Maya: Kaggle (T4)** — fine-tuned directly in Notebook 1 session using Iikim Translator corpus

**Key finding:** `bzd_Latn` (Bribri) and `yua_Latn` (Maya Yucateco) are absent from the base NLLB-200 vocabulary — `tokenizer.convert_tokens_to_ids('bzd_Latn')` returns id 3 (`<s>`), causing the base model to silently output English. Both tokens are added as new special tokens before training.

**Maya note:** Because Maya had no AmericasNLP 2023 data, the dev set image captions (~50 pairs) were upsampled 20× and combined with the Iikim Translator corpus (~11.8k pairs) to give the model exposure to the image caption domain.

## 1. Check GPU

In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if "failed" in gpu_info:
    print("Not connected to a GPU")
else:
    print(gpu_info)

## 2. Install dependencies and mount Drive

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate huggingface_hub

import os, gc, json, torch, warnings, subprocess, unicodedata, glob
import pandas as pd
import shutil
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, logging as hf_logging,
)
from tqdm import tqdm
from huggingface_hub import login
# NOTE: Sections 1-7 run on Google Colab Pro (A100).
# Section 8 (Maya) runs on Kaggle — skip the Colab-specific cells below.
from google.colab import drive

drive.mount("/content/drive")

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
login(token="YOUR_HF_TOKEN")

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 3. Verify NLLB vocabulary coverage

Before training, confirm which language tokens are present in the base vocabulary. `bzd_Latn` (Bribri) and `yua_Latn` (Maya) return id=3, meaning they are absent and will be added as new special tokens.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")

tokens_to_check = ["grn_Latn", "hch_Latn", "nah_Latn", "bzd_Latn", "yua_Latn"]

print("NLLB-200 vocabulary coverage:")
print("="*50)
for token in tokens_to_check:
    token_id = tokenizer.convert_tokens_to_ids(token)
    status = "OK" if token_id != 3 else "ABSENT (returns <s>, id=3)"
    print(f"  {token}: id={token_id} — {status}")
print()
print("bzd_Latn and yua_Latn must be added as new tokens before fine-tuning.")

## 4. Configuration — Wixárika, Nahuatl, Bribri, Guaraní

Maya is fine-tuned separately (see Section 8) because it uses a different corpus, different training platform (Kaggle), and different hyperparameters.

In [ ]:
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
BASE_DIR     = "/content"
OUT_DIR      = f"{BASE_DIR}/nllb-finetuned"
DRIVE_BACKUP = "/content/drive/MyDrive/nllb-finetuned"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DRIVE_BACKUP, exist_ok=True)

# Maya is excluded here — it was fine-tuned in a separate Kaggle session
# using the Iikim Translator corpus (see Section 8 below).
LANG_CONFIG = {
    "guarani":  {"nllb_code": "grn_Latn", "fine_tune": True},
    "wixarika": {"nllb_code": "hch_Latn", "fine_tune": True},
    "nahuatl":  {"nllb_code": "nah_Latn", "fine_tune": True},
    "bribri":   {"nllb_code": "bzd_Latn", "fine_tune": True},
}

LANG_EXT = {
    "guarani": "gn", "wixarika": "hch",
    "nahuatl": "nah", "bribri": "bzd",
}

def normalize(text):
    """NFC normalize and strip whitespace."""
    if not isinstance(text, str):
        return ""
    return " ".join(unicodedata.normalize("NFC", text).split()).strip()

## 5. Training data loaders

In [ ]:
def load_americas_nlp(lang, spa, tgt):
    """AmericasNLP 2023 parallel corpus (base corpus for all languages)."""
    repo = f"{BASE_DIR}/americasnlp2023"
    if not os.path.exists(repo):
        subprocess.run(
            f"git clone https://github.com/AmericasNLP/americasnlp2023.git {repo}",
            shell=True, check=True,
        )
    folder = f"{repo}/data/{lang}-spanish"
    f_spa  = f"{folder}/train.es"
    f_tgt  = f"{folder}/train.{LANG_EXT.get(lang, lang)}"
    if os.path.exists(f_spa) and os.path.exists(f_tgt):
        with open(f_spa, encoding="utf-8") as f:
            spa += [normalize(l) for l in f]
        with open(f_tgt, encoding="utf-8") as f:
            tgt += [normalize(l) for l in f]
        print(f"  [{lang}] AmericasNLP 2023: {len(spa)} pairs")
    else:
        print(f"  [{lang}] AmericasNLP 2023 not found at {folder}")
    return spa, tgt


def load_wixarika(spa, tgt):
    """Pywirrarika corpus (Wixárika)."""
    repo = f"{BASE_DIR}/wixarika-spanish"
    if not os.path.exists(repo):
        subprocess.run(
            f"git clone https://github.com/pywirrarika/wixarika-spanish.git {repo}",
            shell=True, check=True,
        )
    for split in ["corp-train", "corp-dev"]:
        fe = f"{repo}/{split}.es"
        fw = f"{repo}/{split}.wix"
        if os.path.exists(fe) and os.path.exists(fw):
            with open(fe, encoding="utf-8") as f:
                es_lines = [normalize(l) for l in f]
            with open(fw, encoding="utf-8") as f:
                wix_lines = [normalize(l) for l in f]
            pairs = [(s, t) for s, t in zip(es_lines, wix_lines) if s and t]
            spa += [p[0] for p in pairs]
            tgt += [p[1] for p in pairs]
    print(f"  [wixarika] Pywirrarika: {len(spa)} pairs total")
    return spa, tgt


def load_nahuatl(spa, tgt):
    """Axolotl corpus via HuggingFace (Nahuatl)."""
    ds = load_dataset("hackathon-pln-es/Axolotl-Spanish-Nahuatl", split="train")
    for row in ds:
        s = normalize(row["sp"])
        t = normalize(row["nah"])
        if s and t:
            spa.append(s)
            tgt.append(t)
    print(f"  [nahuatl] Axolotl: {len(spa)} pairs total")
    return spa, tgt


def load_bribri(spa, tgt):
    """Feldman & Coto-Solano (2020) Bribri corpus.
    https://aclanthology.org/2020.wildre-1.7
    """
    repo = f"{BASE_DIR}/bribri-coling2020"
    if not os.path.exists(repo):
        subprocess.run(
            f"git clone https://github.com/rolandocoto/bribri-coling2020.git {repo}",
            shell=True, check=True,
        )
    csv_path = f"{repo}/bri-spa-pair-sample-20201101.csv"
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8")
    for _, row in df.iterrows():
        s = normalize(row["Spanish"])
        t = normalize(row["Bribri (Nasal as line)"])
        if s and t:
            spa.append(s)
            tgt.append(t)
    print(f"  [bribri] Feldman corpus: {len(spa)} pairs total")
    return spa, tgt


def load_training_data(lang):
    """Load and combine all training corpora for a given language."""
    spa, tgt = [], []
    spa, tgt = load_americas_nlp(lang, spa, tgt)
    if lang == "wixarika":
        spa, tgt = load_wixarika(spa, tgt)
    if lang == "nahuatl":
        spa, tgt = load_nahuatl(spa, tgt)
    if lang == "bribri":
        spa, tgt = load_bribri(spa, tgt)
    pairs = list({(s, t) for s, t in zip(spa, tgt) if s and t})
    print(f"  [{lang}] TOTAL after dedup: {len(pairs)} pairs")
    return Dataset.from_dict({
        "spanish": [p[0] for p in pairs],
        "target":  [p[1] for p in pairs],
    })

## 6. Fine-tuning loop — Wixárika, Nahuatl, Bribri, Guaraní

Hyperparameters:
- 3 epochs, effective batch 32 (1 × 32 gradient accumulation)
- lr 2e-5, 200 warmup steps, weight_decay 0.01
- Adafactor, fp16=False, gradient checkpointing
- max sequence length 64

Each model saved to `/content/nllb-finetuned/{lang}_final` and backed up to Drive.

In [ ]:
TRAIN_ARGS = dict(
    num_train_epochs             = 3,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 32,   # effective batch = 32
    learning_rate                = 2e-5,
    warmup_steps                 = 200,
    weight_decay                 = 0.01,
    optim                        = "adafactor",
    fp16                         = False,
    gradient_checkpointing       = True,
    save_strategy                = "no",
    logging_steps                = 50,
    predict_with_generate        = True,
)

MAX_SRC_LEN = 64
MAX_TGT_LEN = 64

for lang, config in LANG_CONFIG.items():
    print(f"\n{chr(61)*50}\nFine-tuning: {lang.upper()}\n{chr(61)*50}")
    torch.cuda.empty_cache(); gc.collect()

    dataset = load_training_data(lang)
    assert len(dataset) > 0, f"Empty dataset for {lang}"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="spa_Latn")

    # Add target language token if absent from base vocabulary
    tgt_code = config["nllb_code"]
    existing_id = tokenizer.convert_tokens_to_ids(tgt_code)
    if existing_id == 3 or existing_id == tokenizer.unk_token_id:
        tokenizer.add_special_tokens({"additional_special_tokens": [tgt_code]})
        new_id = tokenizer.convert_tokens_to_ids(tgt_code)
        print(f"  Added {tgt_code} as new token (id {new_id})")
    tokenizer.tgt_lang = tgt_code

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["spanish"], max_length=MAX_SRC_LEN,
            truncation=True, padding=False,
        )
        labels = tokenizer(
            text_target=batch["target"], max_length=MAX_TGT_LEN,
            truncation=True, padding=False,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized = dataset.map(
        tokenize, batched=True, remove_columns=dataset.column_names
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME, low_cpu_mem_usage=True
    ).to("cuda")
    model.resize_token_embeddings(len(tokenizer))

    lang_out_dir = f"{OUT_DIR}/{lang}"
    args     = Seq2SeqTrainingArguments(output_dir=lang_out_dir, **TRAIN_ARGS)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
    trainer  = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=tokenized,
        data_collator=collator,
        tokenizer=tokenizer,
    )

    trainer.train()

    # Save model
    final_path = f"{lang_out_dir}_final"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"  Saved: {final_path}")

    # Backup to Drive
    drive_path = f"{DRIVE_BACKUP}/{lang}_final"
    shutil.copytree(final_path, drive_path, dirs_exist_ok=True)
    print(f"  Drive backup: {drive_path}")

    del model, trainer, tokenized, dataset
    torch.cuda.empty_cache(); gc.collect()

## 7. Verify saved models

In [ ]:
for lang in LANG_CONFIG:
    path = f"{OUT_DIR}/{lang}_final"
    if os.path.exists(path):
        size = sum(os.path.getsize(os.path.join(path, f))
                   for f in os.listdir(path)) / 1e6
        print(f"OK  {lang}: {size:.0f} MB — {path}")
    else:
        print(f"MISSING  {lang}: {path}")

## 8. Maya Yucateco fine-tuning (Kaggle / T4)

Maya was fine-tuned separately on Kaggle because:
- No AmericasNLP 2023 data exists for Maya
- The Iikim Translator corpus was already available on Kaggle
- `yua_Latn` is absent from the base NLLB vocabulary (returns id=3)

**Data strategy:** Iikim Translator train split (~11.8k pairs) + AmericasNLP 2026 dev captions repeated 20× to expose the model to the image caption domain.

**Hyperparameters differ from other languages:**
- 5 epochs with early stopping (patience=2)
- Batch size 4 × 4 grad accumulation (effective batch=16)
- fp16=True (T4 GPU)
- Eval strategy: per epoch, load best model at end

Run this section on **Kaggle** (not Colab), with the task repo already cloned and the Iikim Translator data at `/kaggle/working/iikim_translator/`.

In [ ]:
# Run on Kaggle — install dependencies
!pip install -q transformers datasets sentencepiece accelerate

In [ ]:
import gc
import torch
import pandas as pd
from transformers import (
    NllbTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from datasets import Dataset

torch.cuda.empty_cache()
gc.collect()

# 1. Tokenizer — add yua_Latn as new special token
print("Loading tokenizer...")
tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
tokenizer.add_special_tokens({
    "additional_special_tokens": list(tokenizer.all_special_tokens) + ["yua_Latn"]
})
tokenizer.src_lang = "spa_Latn"
maya_token_id = tokenizer.convert_tokens_to_ids("yua_Latn")
print(f"yua_Latn id: {maya_token_id}, vocab size: {len(tokenizer)}")
assert maya_token_id != 3, "yua_Latn was not added correctly!"

# 2. Model
print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M", low_cpu_mem_usage=True
)
model.resize_token_embeddings(len(tokenizer))
model.gradient_checkpointing_enable()
model = model.to("cuda")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# 3. Data
# Iikim Translator corpus — main training data (~11.8k pairs)
base = "/kaggle/working/iikim_translator/data/mayan/maya-spanish"
with open(f"{base}/train.spanish") as f:
    es_iikim = [l.strip() for l in f if l.strip()]
with open(f"{base}/train.maya") as f:
    maya_iikim = [l.strip() for l in f if l.strip()]

# Iikim dev split — used as validation set
with open(f"{base}/dev.spanish") as f:
    es_dev = [l.strip() for l in f if l.strip()]
with open(f"{base}/dev.maya") as f:
    maya_dev = [l.strip() for l in f if l.strip()]

# AmericasNLP 2026 dev captions — upsampled 20x for domain adaptation
# (image caption style is very different from general parallel text)
dev_2026 = pd.read_json(
    "/kaggle/working/americasnlp2026/data/dev/maya/maya.jsonl", lines=True
)
with open("/kaggle/working/americasnlp2026/baseline/output/maya_spanish_captions.txt") as f:
    es_2026 = [l.strip() for l in f if l.strip()]
maya_2026 = dev_2026["target_caption"].tolist()

REPEAT = 20  # upsample dev captions to expose model to caption domain
es_combined   = es_iikim   + es_2026 * REPEAT
maya_combined = maya_iikim + maya_2026 * REPEAT
print(f"Train pairs: {len(es_combined)} | Dev pairs: {len(es_dev)}")

train_data = Dataset.from_dict({"spanish": es_combined, "maya": maya_combined})
dev_data   = Dataset.from_dict({"spanish": es_dev,      "maya": maya_dev})

# 4. Tokenize
def preprocess(examples):
    tokenizer.src_lang = "spa_Latn"
    model_inputs = tokenizer(
        examples["spanish"], max_length=128, truncation=True, padding=False
    )
    labels = tokenizer(
        text_target=examples["maya"], max_length=128, truncation=True, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_data.map(preprocess, batched=True, remove_columns=["spanish", "maya"])
dev_tok   = dev_data.map(preprocess,   batched=True, remove_columns=["spanish", "maya"])
print("Data tokenized")

# 5. Train
args = Seq2SeqTrainingArguments(
    output_dir                = "/kaggle/working/maya_nllb",
    num_train_epochs          = 5,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,     # effective batch = 16
    warmup_steps              = 200,
    learning_rate             = 2e-5,
    fp16                      = True,    # T4 supports fp16
    eval_strategy             = "epoch",
    save_strategy             = "epoch",
    load_best_model_at_end    = True,
    metric_for_best_model     = "eval_loss",
    predict_with_generate     = False,
    logging_steps             = 100,
    report_to                 = "none",
    optim                     = "adafactor",
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer  = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = train_tok,
    eval_dataset  = dev_tok,
    processing_class = tokenizer,
    data_collator = collator,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Training Maya model...")
trainer.train()

# Save — delete checkpoints first to free disk space
import glob
for cp in glob.glob("/kaggle/working/maya_nllb/checkpoint-*"):
    import shutil; shutil.rmtree(cp)

trainer.model.save_pretrained("/kaggle/working/maya_final")
tokenizer.save_pretrained("/kaggle/working/maya_final")
print("Model saved: /kaggle/working/maya_final")

## 9. Maya: generate test submission

Runs immediately after training while the model is still in memory.

In [ ]:
import json
from collections import Counter

def is_genuine_loop(text, loop_threshold=0.4):
    """Detect degenerate repetition loops.
    Returns True if a single content token > 40% of all tokens.
    """
    tokens = text.split()
    if len(tokens) < 10:
        return False
    content_counts = {t: c for t, c in Counter(tokens).items() if len(t) > 2}
    if not content_counts:
        return False
    top_token = max(content_counts, key=content_counts.get)
    return (content_counts[top_token] / len(tokens)) > loop_threshold

# Use trainer.model directly (already in memory)
model = trainer.model
model.eval()
maya_token_id = tokenizer.convert_tokens_to_ids("yua_Latn")

# Load test data
test_df = pd.read_json(
    "/kaggle/working/americasnlp2026/data/test/maya/maya.jsonl", lines=True
)
with open("/kaggle/working/maya_test_spanish.txt") as f:
    test_spanish = [l.strip() for l in f if l.strip()]

print(f"Test: {len(test_df)} entries, {len(test_spanish)} Spanish captions")
assert len(test_df) == len(test_spanish), "Mismatch between test JSONL and Spanish captions!"

gen_params = {
    "forced_bos_token_id": maya_token_id,
    "max_length":          80,
    "num_beams":           4,
    "repetition_penalty":  1.3,
    "no_repeat_ngram_size": 3,
    "early_stopping":      True,
}

predictions = []
for i in tqdm(range(0, len(test_spanish), 4), desc="Translating Maya test set"):
    batch = test_spanish[i:i+4]
    tokenizer.src_lang = "spa_Latn"
    inputs = tokenizer(
        batch, return_tensors="pt",
        truncation=True, max_length=128, padding=True
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, **gen_params)
    predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    torch.cuda.empty_cache()

# Quality check
loops  = sum(1 for p in predictions if is_genuine_loop(p))
unique = len(set(predictions))
print(f"Total: {len(predictions)}, Unique: {unique}, Loops detected: {loops}")
for i in range(3):
    print(f"  [{test_df.iloc[i]["id"]}]: {predictions[i][:100]}")

# Write submission JSONL
output_file = "/kaggle/working/maya_submission.jsonl"
with open(output_file, "w") as f:
    for i, row in test_df.iterrows():
        entry = row.to_dict()
        entry["predicted_caption"] = predictions[i]
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
print(f"Submission saved: {output_file}")